# K-Means Clustering Example (Mall Customers Dataset)

Here it is demonstrated how to use the `KMeans` module from the CMOR-438 library to discover customer segments.
In this example, the Mall Customers dataset is used to identify natural spending groups without any labelled data.

**Goal: Discover distinct customer segments based on Age, Annual Income, and Spending Score.**

The Mall Customers dataset has:
- **Samples:** 200 mall shoppers
- **Features:** Age, Annual Income (k$), Spending Score (1–100)
- **Labels:** None — this is unsupervised learning

## 1. Setup and Data Loading

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('.')))
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv('../../../data/Mall_Customers.csv')
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

print(f"Dataset loaded: {mall.shape[0]} samples, {len(MALL_FEATURES)} features.")
print(mall[MALL_FEATURES].describe().round(1).to_string())

## 2. Preprocessing

Standardise all three features to zero mean and unit variance.
K-Means uses Euclidean distance, so features with larger scales would otherwise dominate.

In [ ]:
X_raw = mall[MALL_FEATURES].values.astype(float)
scaler = StandardScaler().fit(X_raw)
X = scaler.transform(X_raw)

print(f"Features standardised — mean: {X.mean(axis=0).round(3)}, std: {X.std(axis=0).round(3)}")

## 3. Choose the Best k

Two methods are used to select the optimal number of clusters:
- **Elbow method** — plot inertia (total within-cluster variance) vs k; look for where the curve bends
- **Silhouette score** — measures how similar each point is to its own cluster vs the next nearest; higher is better

In [ ]:
inertias, silhouettes = [], []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(k=k, init='k-means++', n_init=5, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(km.silhouette_score(X))

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'Best K by silhouette: {best_k}  (score={max(silhouettes):.4f})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', lw=1.5, ms=6)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('K-Means - Elbow Method', fontweight='bold')
axes[1].plot(list(k_range), silhouettes, 's-', color='darkorange', lw=1.5, ms=6)
axes[1].axvline(best_k, color='red', linestyle='--', lw=1.2, label=f'Best k={best_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('K-Means - Silhouette Score', fontweight='bold')
axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Fit Final Model

Fit K-Means with the optimal k using 10 random restarts; the best result (lowest inertia) is kept.

In [ ]:
km_final = KMeans(k=best_k, init='k-means++', n_init=10, random_state=42).fit(X)
labels = km_final.labels_
cents_orig = scaler.inverse_transform(km_final.centroids_)
print(f'Inertia: {km_final.inertia_:.2f}  Silhouette: {km_final.silhouette_score(X):.4f}')

## 5. Results and Visualisation

Three pairwise scatter plots show the clusters in original feature space from every angle.
Black X markers show the cluster centroids. The cluster profile table summarises the average characteristics of each group.

In [ ]:
cmap = plt.cm.get_cmap('tab10', best_k)
pairs = [(0,1,'Age','Annual Income (k$)'),(1,2,'Annual Income (k$)','Spending Score (1-100)'),(0,2,'Age','Spending Score (1-100)')]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (i, j, xl, yl) in zip(axes, pairs):
    for c in range(best_k):
        mask = labels==c
        ax.scatter(X_raw[mask,i], X_raw[mask,j], color=cmap(c), s=30, alpha=0.75, label=f'Cluster {c}')
    ax.scatter(cents_orig[:,i], cents_orig[:,j], marker='X', s=200, c='black', zorder=5)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(f'{xl} vs {yl}', fontweight='bold')
handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(handles, lbls, loc='lower center', ncol=best_k, frameon=False, bbox_to_anchor=(0.5,-0.05))
fig.suptitle(f'K-Means Clusters (k={best_k}) - Mall Customers', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

mall['Cluster'] = labels
print("\nCluster Profiles:")
print(mall.groupby('Cluster')[MALL_FEATURES].mean().round(1).to_string())